# Ant Colony Optimization — Traveling Salesman Problem (TSP)

**Tugas Kecerdasan Buatan** — Reguler Malam B1 25/26

**Kasus:** Cari rute terpendek pada graph *Gambar 1* yang **berangkat dari titik H**, **berakhir di titik D**, dan **wajib melewati titik `#`**, menggunakan **Ant Colony Optimization (ACO)**.

Karena graph bersifat *sparse* (tidak semua titik terhubung langsung), jarak antar-titik dihitung dengan algoritma *shortest-path* (**Floyd-Warshall**). Rute hasil ACO lalu dijabarkan kembali menjadi jalur fisik *edge-per-edge* pada graph asli.

Notebook ini *self-contained*: cukup **Runtime → Run all**. Tidak perlu meng-upload file `.py` apa pun.

---
**Referensi algoritma:** M. Dorigo, *Ant Colony Optimization* (aturan transisi probabilistik + pembaruan feromon dengan penguapan).

## 1. Definisi Graph (Gambar 1) + Floyd-Warshall

Mendefinisikan simpul, sisi beserta bobotnya, koordinat untuk visualisasi, lalu menghitung matriks jarak terpendek antar semua pasang titik.

In [ ]:
import math
from typing import Dict, List, Tuple

# Daftar simpul (node) pada Gambar 1.
NODES: List[str] = ["#", "A", "B", "C", "D", "E", "F", "G", "H"]

# Daftar sisi (edge) beserta bobotnya, dibaca langsung dari Gambar 1.
EDGES: List[Tuple[str, str, int]] = [
    ("#", "A", 3),
    ("#", "C", 2),
    ("#", "G", 5),
    ("A", "C", 6),
    ("B", "C", 9),
    ("B", "D", 8),
    ("C", "F", 4),
    ("D", "E", 7),
    ("D", "H", 9),
    ("E", "F", 2),
    ("E", "G", 1),
    ("E", "H", 1),
    ("G", "H", 3),
]

# Koordinat (x, y) untuk visualisasi, mendekati tata letak pada Gambar 1.
COORDS: Dict[str, Tuple[float, float]] = {
    "#": (2.1, 3.0),
    "G": (6.2, 3.0),
    "A": (0.4, 2.0),
    "C": (1.9, 1.6),
    "F": (3.5, 2.3),
    "E": (4.8, 1.7),
    "B": (0.8, 0.5),
    "D": (3.25, 0.9),
    "H": (6.0, 0.4),
}


def adjacency() -> Dict[str, Dict[str, int]]:
    """Bangun adjacency list (graph tak-berarah) dari EDGES."""
    adj: Dict[str, Dict[str, int]] = {n: {} for n in NODES}
    for u, v, w in EDGES:
        adj[u][v] = w
        adj[v][u] = w
    return adj


def floyd_warshall():
    """Jarak terpendek antar semua pasang titik + tabel next-hop."""
    adj = adjacency()
    dist = {u: {v: math.inf for v in NODES} for u in NODES}
    nxt = {u: {v: None for v in NODES} for u in NODES}

    for u in NODES:
        dist[u][u] = 0.0
        nxt[u][u] = u
    for u in NODES:
        for v, w in adj[u].items():
            dist[u][v] = float(w)
            nxt[u][v] = v

    for k in NODES:
        for i in NODES:
            for j in NODES:
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    nxt[i][j] = nxt[i][k]
    return dist, nxt


def reconstruct_path(u: str, v: str, nxt) -> List[str]:
    """Rekonstruksi jalur fisik terpendek u->v lewat tabel next-hop."""
    if nxt[u][v] is None:
        return []
    path = [u]
    while u != v:
        u = nxt[u][v]
        path.append(u)
    return path


def expand_tour(tour: List[str], nxt) -> List[str]:
    """Ubah urutan kunjungan titik menjadi jalur fisik edge-per-edge."""
    if not tour:
        return []
    walk = [tour[0]]
    for a, b in zip(tour, tour[1:]):
        seg = reconstruct_path(a, b, nxt)
        walk.extend(seg[1:])
    return walk


# Tampilkan matriks jarak terpendek antar titik.
dist, nxt = floyd_warshall()
print("Matriks jarak terpendek antar titik:")
print("    " + "".join(f"{n:>5}" for n in NODES))
for u in NODES:
    row = "".join(f"{dist[u][v]:>5.0f}" for v in NODES)
    print(f"{u:>3} {row}")

## 2. Algoritma Ant Colony Optimization (ACO)

Implementasi ACO untuk *open-path TSP* dengan titik awal (H) dan akhir (D) tetap. Setiap semut membangun rute `H → (semua titik perantara) → D`, sehingga `#` otomatis ikut dikunjungi. Feromon diperbarui dengan penguapan + deposit berbanding terbalik dengan panjang rute, plus penguatan *elitist* pada rute terbaik global.

In [ ]:
import itertools
import random
from dataclasses import dataclass, field


@dataclass
class ACOParams:
    n_ants: int = 20          # jumlah semut per iterasi
    n_iterations: int = 200   # jumlah iterasi
    alpha: float = 1.0        # bobot pengaruh feromon
    beta: float = 3.0         # bobot pengaruh heuristik (1/jarak)
    rho: float = 0.5          # laju penguapan feromon
    q: float = 100.0          # konstanta deposit feromon
    tau0: float = 1.0         # feromon awal
    seed: int = 42            # seed RNG agar hasil reproducible


@dataclass
class ACOResult:
    tour: List[str]
    length: float
    walk: List[str]
    history: List[float] = field(default_factory=list)


class AntColonyTSP:
    """ACO untuk TSP open-path dengan titik awal & akhir tetap."""

    def __init__(self, start="H", end="D", params=None):
        self.start = start
        self.end = end
        self.params = params or ACOParams()
        self.dist, self.nxt = floyd_warshall()
        self.intermediates = [n for n in NODES if n not in (start, end)]
        self.tau = {u: {v: self.params.tau0 for v in NODES} for u in NODES}
        self.rng = random.Random(self.params.seed)

    def _eta(self, u, v):
        d = self.dist[u][v]
        return 1.0 / d if d > 0 else 0.0

    def _pick_next(self, current, candidates):
        """Pilih titik berikutnya secara probabilistik (transisi ACO)."""
        weights = []
        for v in candidates:
            tau = self.tau[current][v] ** self.params.alpha
            eta = self._eta(current, v) ** self.params.beta
            weights.append(tau * eta)
        total = sum(weights)
        if total <= 0:
            return self.rng.choice(candidates)
        r = self.rng.uniform(0, total)
        upto = 0.0
        for v, w in zip(candidates, weights):
            upto += w
            if upto >= r:
                return v
        return candidates[-1]

    def _build_tour(self):
        tour = [self.start]
        unvisited = list(self.intermediates)
        current = self.start
        while unvisited:
            nx = self._pick_next(current, unvisited)
            tour.append(nx)
            unvisited.remove(nx)
            current = nx
        tour.append(self.end)
        return tour, self._tour_length(tour)

    def _tour_length(self, tour):
        return sum(self.dist[a][b] for a, b in zip(tour, tour[1:]))

    def _update_pheromone(self, ant_tours):
        for u in NODES:
            for v in NODES:
                self.tau[u][v] *= (1.0 - self.params.rho)
        for tour, length in ant_tours:
            if length <= 0:
                continue
            deposit = self.params.q / length
            for a, b in zip(tour, tour[1:]):
                self.tau[a][b] += deposit
                self.tau[b][a] += deposit

    def run(self):
        best_tour, best_len, history = [], float("inf"), []
        for _ in range(self.params.n_iterations):
            ant_tours = [self._build_tour() for _ in range(self.params.n_ants)]
            for tour, length in ant_tours:
                if length < best_len:
                    best_len, best_tour = length, tour
            self._update_pheromone(ant_tours)
            if best_tour:  # reinforcement elitist
                deposit = self.params.q / best_len
                for a, b in zip(best_tour, best_tour[1:]):
                    self.tau[a][b] += deposit
                    self.tau[b][a] += deposit
            history.append(best_len)
        walk = expand_tour(best_tour, self.nxt)
        return ACOResult(tour=best_tour, length=best_len, walk=walk, history=history)

    def brute_force_optimum(self):
        """Solusi optimal eksak via brute force (untuk verifikasi)."""
        best_tour, best_len = [], float("inf")
        for perm in itertools.permutations(self.intermediates):
            tour = [self.start, *perm, self.end]
            length = self._tour_length(tour)
            if length < best_len:
                best_len, best_tour = length, tour
        return best_tour, best_len

## 3. Jalankan ACO + Verifikasi (Brute Force)

Menjalankan solver dan membandingkan hasil ACO dengan solusi optimal eksak (brute force atas seluruh permutasi titik perantara).

In [ ]:
solver = AntColonyTSP(start="H", end="D", params=ACOParams())
result = solver.run()

print("=" * 60)
print("  ANT COLONY OPTIMIZATION - TSP (Gambar 1)")
print("  Rute dari H ke D, wajib melewati '#'")
print("=" * 60)
print(f"\nRute ACO (urutan titik): {' -> '.join(result.tour)}")
print(f"Total jarak             : {result.length:.0f}")
print(f"Jalur fisik (edge graph): {' -> '.join(result.walk)}")
print(f"'#' dilewati             : {'YA' if '#' in result.tour else 'TIDAK'}")

bf_tour, bf_len = solver.brute_force_optimum()
print("\n--- Verifikasi (brute force) ---")
print(f"Rute optimal            : {' -> '.join(bf_tour)}")
print(f"Total jarak optimal     : {bf_len:.0f}")
status = "OPTIMAL" if abs(result.length - bf_len) < 1e-9 else "BELUM OPTIMAL"
print(f"Status hasil ACO        : {status}")

## 4. Visualisasi Graph + Rute Terbaik

Menggambar graph *Gambar 1* (sisi abu-abu + bobot) dan menandai jalur fisik rute terbaik hasil ACO dengan panah oranye. `#` merah, titik awal/akhir (H/D) hijau, sisanya ungu.

In [ ]:
import matplotlib.pyplot as plt

walk = result.walk
fig, ax = plt.subplots(figsize=(10, 5))

# Semua edge graph (abu-abu) + label bobot.
for u, v, w in EDGES:
    x1, y1 = COORDS[u]
    x2, y2 = COORDS[v]
    ax.plot([x1, x2], [y1, y2], color="#bbbbbb", lw=2, zorder=1)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(w),
            fontsize=10, color="#444444", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none"))

# Rute terbaik (jalur fisik) = panah oranye tebal.
for a, b in zip(walk, walk[1:]):
    x1, y1 = COORDS[a]
    x2, y2 = COORDS[b]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="#e8590c",
                                lw=3, shrinkA=14, shrinkB=14), zorder=2)

# Simpul.
for n, (x, y) in COORDS.items():
    is_endpoint = n in (solver.start, solver.end)
    is_hash = n == "#"
    color = "#c0392b" if is_hash else ("#2f9e44" if is_endpoint else "#6741d9")
    ax.scatter([x], [y], s=900, color=color, zorder=3, edgecolors="white", linewidths=2)
    ax.text(x, y, n, color="white", fontsize=13, fontweight="bold",
            ha="center", va="center", zorder=4)

ax.set_title(f"Rute ACO: {' -> '.join(result.tour)}  |  Total jarak = {result.length:.0f}", fontsize=12)
ax.axis("off")
fig.tight_layout()
fig.savefig("hasil_rute.png", dpi=130)
plt.show()
print(f"Rute  : {' -> '.join(result.tour)}  (jarak {result.length:.0f})")
print(f"Jalur : {' -> '.join(walk)}")

## 5. Grafik Konvergensi

Menampilkan bagaimana jarak rute terbaik menurun (konvergen) sepanjang iterasi — ciri khas perilaku ACO.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(result.history) + 1), result.history, color="#1971c2", lw=2)
plt.xlabel("Iterasi")
plt.ylabel("Jarak rute terbaik")
plt.title("Konvergensi ACO")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()